# Translation of data - NL2EN

In [1]:
%pip install transformers accelerate
%pip install ctranslate2 OpenNMT-py==2.* sentencepiece
%pip install sacremoses

     |████████████████████████████████| 330 kB 1.7 MB/s eta 0:00:01
Note: you may need to restart the kernel to use updated packages.
zsh:1: no matches found: OpenNMT-py==2.*
Note: you may need to restart the kernel to use updated packages.
     |████████████████████████████████| 897 kB 8.4 MB/s eta 0:00:01
Note: you may need to restart the kernel to use updated packages.


In [1]:
#%pip install transformers -U
import transformers

In [17]:
%pip install sentencepiece

Note: you may need to restart the kernel to use updated packages.


In [14]:
#https://opennmt.net/CTranslate2/guides/transformers.html#marianmt
!ct2-transformers-converter --model Helsinki-NLP/opus-mt-nl-en --output_dir /workspace/mijnidbcoachnlp/data/ct2_model --force

2025-02-08 12:36:03.294782: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
source.spm: 100%|████████████████████████████| 814k/814k [00:00<00:00, 9.10MB/s]
target.spm: 100%|████████████████████████████| 790k/790k [00:00<00:00, 3.20MB/s]
vocab.json: 100%|██████████████████████████| 1.66M/1.66M [00:00<00:00, 18.8MB/s]


In [4]:
# Load dataset
import pandas as pd
import tqdm

path = "/workspace/mijnidbcoachnlp/data/analysis_data/sentence_data_new.xlsx"
df = pd.read_excel(path, index_col=0)
df.tail()

,Message_ID,Sentence,Clean_Sentence,Sentence_ID
41138,32679,"Ze hebben de urine op kweek gezet, kan morgenv...","Ze hebben de urine op kweek gezet, kan morgenv...",41139
41139,32679,Afgelopen jaren is er al vaker bloed in mijn u...,Afgelopen jaren is er al vaker bloed in mijn u...,41140
41140,32679,Wat adviseert u hiermee te doen?,Wat adviseert u hiermee te doen?,41141
41141,32681,"Hoi [PERSOON], Oke, dat is iig al een gerustst...","Oke, dat is iig al een geruststelling ik wach...",41142
41142,32681,Dankjewel voor de snelle reactie!,Dankjewel voor de snelle reactie!,41143


In [5]:
messages = df["Sentence"].tolist()

In [6]:
import ctranslate2
import transformers
import tqdm

# Load CTranslate2 Translator and the tokenizer
translator = ctranslate2.Translator('/workspace/mijnidbcoachnlp/data/ct2_model')
tokenizer = transformers.AutoTokenizer.from_pretrained("Helsinki-NLP/opus-mt-nl-en")

# Function to translate a batch of messages using CTranslate2
def translate_batch(messages, batch_size=10):
    translations = []
    
    for i in tqdm.tqdm(range(0, len(messages), batch_size)):
        batch = messages[i:i + batch_size]
        
        # Tokenize the batch of messages into tokens (lists of token strings)
        source_batch = [tokenizer.convert_ids_to_tokens(tokenizer.encode(message, add_special_tokens=True)) for message in batch]

        # Translate the batch using CTranslate2 (expects a list of tokenized inputs)
        results = translator.translate_batch(source_batch)

        # Decode the hypotheses (translations) back to text
        for result in results:
            translated_tokens = result.hypotheses[0]  # Access the first hypothesis (most likely translation)
            translated_text = tokenizer.convert_tokens_to_string(translated_tokens)  # Convert tokens back to string
            translations.append(translated_text)
        
        # Print original and translated messages
        #for original, translated in zip(batch, translations[-len(batch):]):
            #print(f"Original: {original}")
            #print(f"Translated: {translated}")
    
    return translations

# Function to clean and filter valid text messages
def clean_messages(messages):
    return [str(message) for message in messages if isinstance(message, str) and message.strip()]

# Clean the messages to remove any non-string or empty values
cleaned_messages = clean_messages(messages)

# Translate messages in batches using CTranslate2
translated_messages = translate_batch(cleaned_messages)

# Add the translations back into the DataFrame (optional)
df["Translated_Sentence"] = [None if not isinstance(msg, str) else translated_messages.pop(0) for msg in messages]

# Save the translated messages to a new file (optional)
df.to_excel("/workspace/mijnidbcoachnlp/data/analysis_data/translated_sentence_data_new.xlsx", index=False)


100%|██████████| 4115/4115 [1:24:02<00:00,  1.23s/it]


In [7]:
# read the message df and map the language to the sentences
df = pd.read_excel("/workspace/mijnidbcoachnlp/data/analysis_data/translated_sentence_data_new.xlsx", index_col=0)
message_df = pd.read_excel("/workspace/mijnidbcoachnlp/data/analysis_data/message_data.xlsx", index_col=0)

df = df.merge(message_df[['Message_ID']], on="Message_ID", how='left')
df

,Message_ID,Sentence,Clean_Sentence,Sentence_ID,Translated_Sentence
0,3,Ik ben 2 weken geleden met spoed opgenomen in ...,Ik ben 2 weken geleden met spoed opgenomen in ...,1,I was rushed into the [PRESSION] two weeks ago...
1,3,"Ik kreeg acuut een pijnlijke druk op de borst,...","Ik kreeg acuut een pijnlijke druk op de borst,...",2,I was acutely under a painful pressure on the ...
2,3,Het begon 1 uur na het avondeten.,Het begon 1 uur na het avondeten.,3,It started 1 hour after dinner.
3,3,"Ik had al de hele dag migraine, had dus ook we...","Ik had al de hele dag migraine, had dus ook we...",4,"I had migraines all day, so I hadn't eaten much."
4,3,"Ik werd heel erg misselijk, braakneigingen, du...","Ik werd heel erg misselijk, braakneigingen, du...",5,"I got very nauseous, vomiting, dizzy and rejuv..."
...,...,...,...,...,...
41138,32679,"Ze hebben de urine op kweek gezet, kan morgenv...","Ze hebben de urine op kweek gezet, kan morgenv...",41139,"They've put the urine on culture, can get the ..."
41139,32679,Afgelopen jaren is er al vaker bloed in mijn u...,Afgelopen jaren is er al vaker bloed in mijn u...,41140,"In recent years, blood has been found in my ur..."
41140,32679,Wat adviseert u hiermee te doen?,Wat adviseert u hiermee te doen?,41141,What do you suggest you do with this?
41141,32681,"Hoi [PERSOON], Oke, dat is iig al een gerustst...","Oke, dat is iig al een geruststelling ik wach...",41142,"Hi [PRESSON], Okay, that's a relief;) I'll wai..."


In [8]:
df.to_excel("/workspace/mijnidbcoachnlp/data/analysis_data/translated_sentence_data_new.xlsx", index=False)
